In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Задаем параметры
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = pd.date_range(start=start_date, end=end_date, freq='D')
np.random.seed(42)

# 2. Создаем базовый DataFrame
df = pd.DataFrame({'date': date_range})
df['day_of_year'] = df['date'].dt.dayofyear
df['month'] = df['date'].dt.month

# 3. Генерируем температуру (только одну!)
year_length = 365
base_temp = -18 * np.cos(2 * np.pi * (df['day_of_year'] - 15) / year_length) + 6
noise_temp = np.random.normal(0, 3, len(df))
df['t_avg_c'] = base_temp + noise_temp

# 4. Генерируем осадки (только исходные, без накопленных)
def generate_precip(day_of_year, temp):
    seasonal_factor = 1 + 0.7 * np.sin(2 * np.pi * (day_of_year - 180) / 365)
    
    if temp < 0:
        prob = 0.3
        magnitude = np.random.gamma(1.5, 2)
    else:
        prob = 0.5
        magnitude = np.random.gamma(2.5, 3)
    
    if np.random.rand() < prob:
        return max(0, magnitude * seasonal_factor)
    return 0

df['precip_mm'] = [generate_precip(d, t) for d, t in zip(df['day_of_year'], df['t_avg_c'])]

# 5. Снеготаяние и снежный покров
snow_pack = 0
snow_pack_history = []
snow_melt_history = []

for temp in df['t_avg_c']:
    if temp < 0:
        snow_pack += max(0, -temp * 0.3)
        snow_melt_history.append(0)
    else:
        if snow_pack > 0:
            melt_rate = min(1.0, temp / 10)
            melt = min(snow_pack, 5 + 15 * melt_rate)
            snow_melt_history.append(melt)
            snow_pack = max(0, snow_pack - melt)
        else:
            snow_melt_history.append(0)
    snow_pack_history.append(snow_pack)

df['snow_pack_mm'] = snow_pack_history
df['snow_melt_mm'] = snow_melt_history

# 6. Испарение (важно!)
def calculate_evaporation(temp, snow_pack, month):
    base_evap = max(0, (temp - 5) * 0.3)
    if snow_pack > 0:
        base_evap *= 0.3
    seasonal_factor = 1 + 0.3 * np.sin(2 * np.pi * (month - 6) / 12)
    noise = np.random.normal(1, 0.2)
    return max(0, base_evap * seasonal_factor * noise)

df['evaporation_mm'] = [
    calculate_evaporation(t, s, m)
    for t, s, m in zip(df['t_avg_c'], df['snow_pack_mm'], df['month'])
]

# 7. Ледовые заторы
def simulate_ice_jams(day_of_year, temp, snow_pack, precip):
    if 60 <= day_of_year <= 120 and snow_pack > 100 and temp > 0:
        ice_jam_factor = min(1.0, (temp / 15) * (snow_pack / 200))
        if precip > 10:
            ice_jam_factor *= 1.5
        if np.random.rand() < ice_jam_factor * 0.7:
            return round(np.random.gamma(2, 15) * ice_jam_factor, 1)
    return 0

df['ice_jam_effect_cm'] = [
    simulate_ice_jams(doy, temp, snow, prec)
    for doy, temp, snow, prec in zip(df['day_of_year'], df['t_avg_c'],
                                        df['snow_pack_mm'], df['precip_mm'])
]

# 8. Базовый уровень (упрощенно)
yearly_factor = {2023: 1.0, 2024: 1.2}
df['year'] = df['date'].dt.year

base_level = []
for idx, row in df.iterrows():
    seasonal = 40 + 70 * np.exp(-((row['day_of_year'] - 150) / 45)**4)
    seasonal *= yearly_factor[row['year']]
    base_level.append(seasonal)

df['base_water_level_cm'] = base_level

# 9. Собираем финальный уровень воды
# Влияние осадков (с задержкой)
rain_effect = np.roll(df['precip_mm'] * 2.0, 2)
rain_effect[:2] = 0

# Влияние снеготаяния
snow_effect = df['snow_melt_mm'] * 0.7

# Влияние испарения (отрицательное)
evaporation_effect = -df['evaporation_mm'] * 0.2

# Влияние заторов
jam_effect = df['ice_jam_effect_cm']

# Собираем
water_level = (df['base_water_level_cm'] + rain_effect +
                snow_effect + evaporation_effect + jam_effect)

# Добавляем авторегрессию (память реки)
water_level_ar = water_level.copy()
for i in range(1, len(water_level_ar)):
    water_level_ar[i] = 0.7 * water_level_ar[i-1] + 0.3 * water_level[i]

# Добавляем шум
noise = np.random.normal(0, 4, len(df))
df['water_level_cm'] = np.maximum(20, water_level_ar + noise).round(1)

# 10. Создаем ЛАГОВЫЕ признаки (только самые важные!)
df['water_level_1day_lag'] = df['water_level_cm'].shift(1).fillna(method='bfill')

# 11. Тренды (важно!)
df['level_trend_3d'] = df['water_level_cm'].diff(3).fillna(0)
df['temp_trend_3d'] = df['t_avg_c'].diff(3).fillna(0)

# 12. Удаляем промежуточные признаки, которые не будем использовать
# (оставляем только то, что нужно для модели)
final_features = [
    'date',
    'water_level_cm',           # Целевая переменная
    'water_level_1day_lag',      # Лаг (важнейший!)
    't_avg_c',                    # Температура
    'precip_mm',                   # Осадки
    'snow_pack_mm',                # Снегозапас
    'snow_melt_mm',                # Снеготаяние
    'evaporation_mm',              # Испарение
    'ice_jam_effect_cm',           # Заторы
    'level_trend_3d',              # Тренд уровня
    'temp_trend_3d'                 # Тренд температуры
]

df_final = df[final_features].copy()

# 13. Проверяем VIF (должен быть низким)
print("="*60)
print("ФИНАЛЬНЫЙ НАБОР ПРИЗНАКОВ:")
print("="*60)
print(f"Всего признаков: {len(df_final.columns) - 1} (без даты и целевой)")
print("\nПризнаки:")
for col in df_final.columns:
    if col not in ['date', 'water_level_cm']:
        print(f"  - {col}")

# 14. Быстрая проверка корреляций
print("\n" + "="*60)
print("КОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ:")
print("="*60)

feature_cols = [c for c in final_features if c not in ['date', 'water_level_cm']]
for col in feature_cols:
    corr = df_final['water_level_cm'].corr(df_final[col])
    print(f"{col:25} → {corr:.3f}")

# 15. Сохраняем
df_final.to_csv('kacha_river_clean.csv', index=False, encoding='utf-8')
print(f"   Строк: {len(df_final)}")
print(f"   Колонок: {len(df_final.columns)}")

# Показываем первые строки
print("\n" + "="*60)
print("ПЕРВЫЕ 5 СТРОК:")
print("="*60)
print(df_final.head())

ФИНАЛЬНЫЙ НАБОР ПРИЗНАКОВ:
Всего признаков: 10 (без даты и целевой)

Признаки:
  - water_level_1day_lag
  - t_avg_c
  - precip_mm
  - snow_pack_mm
  - snow_melt_mm
  - evaporation_mm
  - ice_jam_effect_cm
  - level_trend_3d
  - temp_trend_3d

КОРРЕЛЯЦИЯ С ЦЕЛЕВОЙ ПЕРЕМЕННОЙ:
water_level_1day_lag      → 0.976
t_avg_c                   → 0.588
precip_mm                 → 0.023
snow_pack_mm              → -0.395
snow_melt_mm              → 0.068
evaporation_mm            → 0.378
ice_jam_effect_cm         → 0.011
level_trend_3d            → 0.127
temp_trend_3d             → 0.081
   Строк: 731
   Колонок: 11

ПЕРВЫЕ 5 СТРОК:
        date  water_level_cm  water_level_1day_lag    t_avg_c  precip_mm  \
0 2023-01-01            40.5                  40.5  -9.989658   0.000000   
1 2023-01-02            33.3                  40.5 -11.965954   5.048294   
2 2023-01-03            35.9                  33.3  -9.674256   0.000000   
3 2023-01-04            41.9                  35.9  -7.109171   0.0

C:\Users\Great_Ded\AppData\Local\Temp\ipykernel_16492\971536283.py:131: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['water_level_1day_lag'] = df['water_level_cm'].shift(1).fillna(method='bfill')
